# OLMo frequency-geometry trajectory (Experiment 2)
Fixes the training-dynamics criticism. For each checkpoint we measure norm-frequency coupling, directional-frequency coupling, baseline-logit frequency coupling, spectral anisotropy and effective rank. The frequency direction is computed once from a reference checkpoint and reused (sign-stable) across training.

**Claim to support:** two *separable* effects — co-evolution builds the directional channel; weight decay suppresses the norm channel. The norm column being present (not `--`) is the whole point.

In [ ]:
# --- environment (pins matching the pipeline) ---
# transformers==4.46.2  numpy==1.26.4  ; PyTorch nightly cu128 on newer instances.
import os, gc, json, math, pathlib
import numpy as np, torch
from tqdm.auto import tqdm
import rw_core as rc          # tested core (rw_core_smoketest.py: 19/19)
import rw_modelio as mio      # model IO / hooks / generation

ART = pathlib.Path(os.environ.get("RW_ART", "artifacts")); ART.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def save(obj, name):
    p = ART / name
    np.savez_compressed(p, **obj) if name.endswith(".npz") else \
        p.write_text(json.dumps(obj, indent=2, default=float))
    print("saved", p)

def exists(name):  # skip-if-exists guard
    return (ART / name).exists()


In [ ]:
# Checkpoint revisions. Fill with the actual HF revision strings for your
# OLMo run (e.g. "stepNNNN-tokensMMMB"). Earlier-than-1k checkpoints are the
# most valuable: they let you see the directional channel *form*.
OLMO_REPO = "allenai/OLMo-2-1124-7B"   # or your run
REVISIONS = [
    # "step500-tokens...B",   # <-- add sub-1k if available
    "main",                   # placeholder; replace with real step revisions
]
REFERENCE_REV = REVISIONS[-1]          # late checkpoint defines the sign of r
N_NEUTRAL = 256                        # lines for baseline-logit h_bar

In [ ]:
# neutral sample for token frequencies + baseline logit (WikiText-103)
from datasets import load_dataset
wt = load_dataset("wikitext", "wikitext-103-raw-v1", split="test")
neutral = [t for t in wt["text"] if len(t) > 40][:N_NEUTRAL]
print(len(neutral), "neutral lines")

In [ ]:
@torch.no_grad()
def load_unembedding_and_hbar(rev):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(OLMO_REPO, revision=rev)
    model = AutoModelForCausalLM.from_pretrained(
        OLMO_REPO, revision=rev, torch_dtype=torch.bfloat16,
        device_map=DEVICE, output_hidden_states=True).eval()
    W, final_norm, bias = mio.get_unembedding(model)            # W on CPU f32
    # h_bar = mean final-normalized residual over neutral sample
    acc = torch.zeros(W.shape[1], dtype=torch.float32)
    n = 0
    for line in neutral:
        ids = tok(line, return_tensors="pt").to(DEVICE)
        hs = model(**ids).hidden_states[-1][0, -1]
        acc += mio.rmsnorm_apply(final_norm, hs.to(model.dtype)).float().cpu()
        n += 1
    h_bar = (acc / n).numpy()
    logf = mio.token_logfreq(tok, neutral, W.shape[0])
    del model; gc.collect(); torch.cuda.empty_cache()
    return W, logf, h_bar, bias

In [ ]:
# reference direction first (sign anchor), then sweep checkpoints
if not exists("olmo_freq_traj.json"):
    Wref, logf_ref, hbar_ref, bias_ref = load_unembedding_and_hbar(REFERENCE_REV)
    r_ref = rc.frequency_direction(Wref, logf_ref)
    rows = []
    for rev in tqdm(REVISIONS):
        W, logf, h_bar, bias = load_unembedding_and_hbar(rev)
        g = rc.freq_geometry(W, logf, r_ref=r_ref, h_bar=h_bar, final_bias=bias)
        g.pop("r")
        g["revision"] = rev
        rows.append(g)
        del W; gc.collect()
    save({"rows": rows}, "olmo_freq_traj.json")
    for row in rows: print(row)

In [ ]:
# plot the two channels on one axis: norm coupling and directional coupling
import matplotlib.pyplot as plt
rows = json.loads((ART/"olmo_freq_traj.json").read_text())["rows"]
x = list(range(len(rows)))
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(x, [r["norm_freq"] for r in rows], "o-", label="norm-freq")
ax[0].plot(x, [r["dir_freq"]  for r in rows], "s-", label="dir-freq")
ax[0].plot(x, [r["base_freq"] for r in rows], "^-", label="base-freq")
ax[0].set_title("frequency coupling vs checkpoint"); ax[0].legend(); ax[0].set_xticks(x)
ax[0].set_xticklabels([r["revision"] for r in rows], rotation=45, ha="right")
ax[1].plot(x, [r["spectral_aniso"] for r in rows], "o-", label="anisotropy")
ax2 = ax[1].twinx(); ax2.plot(x, [r["eff_rank"] for r in rows], "s--", color="tab:red", label="eff_rank")
ax[1].set_title("spectral anisotropy / effective rank")
plt.tight_layout(); plt.savefig(ART/"olmo_freq_traj.png", dpi=140); plt.show()

### Reporting language (use whichever the data shows)
> Two effects are separable. Directional frequency coupling is high and stable across training. Norm-based frequency coupling [stays low / declines] over the same checkpoints. The directional channel does not require weight decay; the norm channel is reduced rather than redirected. Replace the toy-model 'migration' label with this.

If norm-freq is already near zero at the earliest checkpoint, say so: the model never used the norm channel, consistent with the toy result that decay suppresses it.